# Self-Reflection & Critique

A model writes a draft. We ask it to **score that draft against a written checklist**. Then we ask it to **rewrite the draft** using what the score said. Repeat until it is good enough, or until we run out of tries.

Everything is in this one notebook. Just plain Python functions.

**You need:** an OpenAI API key. A full run costs about one cent.

**How to use it:** Runtime → Run all. The last cell asks you to paste a draft.

## Step 1 — Install and import

Two libraries do the heavy lifting:

- **`openai`** — talks to the model.
- **`json`** — turns the model's reply into a Python dictionary. It comes with Python; nothing to install.

In [ ]:
%pip install -q openai

import json
import textwrap

from openai import OpenAI

print("Ready.")

## Step 2 — Your API key

Either add a Colab secret called `OPENAI_API_KEY` (the 🔑 icon on the left, switch on *Notebook access*), or just run this cell and paste the key when it asks. **`getpass`** hides it as you type.

In [ ]:
import getpass
import os

key = os.environ.get("OPENAI_API_KEY")

if not key:
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        key = None

if not key:
    key = getpass.getpass("Paste your OpenAI API key (hidden): ").strip()

client = OpenAI(api_key=key)

MODEL = "gpt-4o-mini"
PASS_MARK = 8
MAX_ROUNDS = 3

print("Connected. Model:", MODEL)

## Step 3 — The rules we judge against

Three pieces of text. They never change during a run, so that every score means the same thing.

- **BRIEF** — what the writing has to do.
- **CHECKLIST** — the five things we score, out of 10 each.
- **PRINCIPLES** — how the critic must behave, on every task.

In [ ]:
BRIEF = """
Write a short launch email for working professionals thinking about a live
weekend course in Generative AI and Agentic AI.

Who reads it:
- Software engineers, data analysts, DevOps people, tech managers
- They are busy, they dislike hype, and they want proof it is practical

Rules:
- 140 to 180 words
- Warm but professional
- Mention live classes, hands-on projects, and why it helps their career
- Do not promise salaries, jobs or placement
- Finish with one clear next step
"""

CHECKLIST = """
Give each of these a score out of 10:

1. Audience fit   - speaks to working professionals, no hype
2. Specificity    - real details, not vague claims
3. Rules followed - word count, tone, and everything the brief asked for
4. Evidence       - no promises it cannot back up
5. Next step      - ends with one clear thing to do
"""

PRINCIPLES = """
- Never invent facts, rankings, salaries or placements.
- Prefer concrete detail over general enthusiasm.
- Remember the reader is busy and hard to impress.
- Say what to change. Never give a vague opinion.
"""

print("Rules loaded.")

## Step 4 — One function to talk to the model

Every call in this notebook goes through `ask_model`.

The important bit is `response_format={"type": "json_object"}`. That is an **OpenAI library feature**: when you switch it on, the reply is guaranteed to be valid JSON. So we can hand it straight to `json.loads` and get a Python dictionary. No cleaning up, no string trimming.

In [ ]:
def ask_model(instructions, question, temperature=0.3, as_json=False):
    """Send one question to the model and return its answer as text."""
    extra = {}
    if as_json:
        # OpenAI guarantees valid JSON when we ask for it this way.
        extra["response_format"] = {"type": "json_object"}

    answer = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
        **extra,
    )
    return answer.choices[0].message.content


print("ask_model is ready.")

## Step 5 — The critic

Ask the model to score the draft and say what to change. We tell it the exact keys we want back, and `json.loads` turns the reply into an ordinary Python dictionary:

```python
{
  "score": 3,
  "summary": "...",
  "issues": [{"criterion": "...", "severity": "major",
              "problem": "...", "fix": "..."}],
  "lesson": "..."
}
```

Nothing clever — a dictionary with a list inside it.

> In a real product you would check those keys with a library like **Pydantic** before trusting them. We are keeping it plain here so the loop stays readable.

In [ ]:
def critique(draft, lessons):
    """Ask the model to score a draft. Returns a plain dictionary."""
    instructions = (
        "You are a strict but fair editor. Judge the draft against the brief, "
        "the checklist and the principles. Never praise vague writing. "
        "Never invent facts. Quote the words at fault."
    )

    earlier = "\n".join("- " + lesson for lesson in lessons) or "Nothing yet."

    question = f"""
BRIEF:
{BRIEF}

CHECKLIST:
{CHECKLIST}

PRINCIPLES:
{PRINCIPLES}

LESSONS FROM EARLIER ROUNDS:
{earlier}

THE DRAFT:
{draft}

Reply with JSON using exactly these keys:
  "score"   - one number out of 10, the lowest of the five checklist scores
  "summary" - one sentence on the state of the draft
  "issues"  - a list of at most 5 problems. Each one has:
                "criterion" - which checklist line it is about
                "severity"  - "minor", "major" or "blocking"
                "problem"   - what is wrong, quoting the draft
                "fix"       - the exact change to make
  "lesson"  - one sentence worth remembering for the next round
"""

    reply = ask_model(instructions, question, temperature=0, as_json=True)
    return json.loads(reply)


print("critique() is ready.")

## Step 6 — The writer

The writer gets the draft and the list of fixes. **It never sees the checklist.** If it knew how it was being scored, it would write to please the score instead of the reader.

In [ ]:
def rewrite(draft, report):
    """Rewrite a draft using the fixes the critic asked for."""
    instructions = (
        "You are a rewriting editor. Apply the changes you are given. "
        "Reply with the rewritten draft only - no notes, no explanation."
    )

    changes = "\n".join("- " + issue["fix"] for issue in report["issues"])

    question = f"""
BRIEF:
{BRIEF}

PRINCIPLES TO KEEP:
{PRINCIPLES}

CURRENT DRAFT:
{draft}

CHANGES TO MAKE:
{changes}
"""

    return ask_model(instructions, question, temperature=0.4).strip()


print("rewrite() is ready.")

## Step 7 — Who decides when to stop

**Our code decides, not the model.** A model that can be talked into one more round, will be.

A draft passes only if it hits the pass mark **and** has no `blocking` problem. A score of 9 with a blocking problem still fails.

In [ ]:
def has_passed(report):
    """Decide if a draft is good enough. This is our decision, not the model's."""
    blocking = [i for i in report["issues"] if i["severity"] == "blocking"]
    return report["score"] >= PASS_MARK and len(blocking) == 0


print("has_passed() is ready.")

## Step 8 — Printing it nicely

Two small helpers. **`textwrap.fill`** is a Python built-in that breaks long text into tidy lines so nothing runs off the screen.

In [ ]:
def heading(left, right=""):
    """Print a title bar."""
    print("\n" + "=" * 70)
    print(left + " " * max(1, 70 - len(left) - len(right)) + right if right else left)
    print("=" * 70)


def paragraph(text):
    """Print text wrapped to 70 characters."""
    for line in str(text).split("\n"):
        print(textwrap.fill(line, width=70) if line.strip() else "")


print("Printing helpers ready.")

## Step 9 — The loop

Score, rewrite, score again. It stops when any one of these is true:

1. the draft passed,
2. we used all our rounds,
3. the score stopped going up.

In [ ]:
def improve(draft):
    """Score and rewrite a draft until it passes or we run out of rounds."""
    heading("STEP 1 - YOUR DRAFT", f"{len(draft.split())} words")
    paragraph(draft)

    lessons = []
    scores = []
    step = 1
    last_score = None

    for round_number in range(1, MAX_ROUNDS + 1):
        report = critique(draft, lessons)
        passed = has_passed(report)
        scores.append(report["score"])
        lessons.append(report["lesson"])

        step += 1
        name = "THE CRITIC SCORES YOUR DRAFT" if last_score is None else "THE CRITIC SCORES THE REWRITE"
        verdict = "PASSED" if passed else f"below {PASS_MARK}"
        moved = "" if last_score is None else f"  ({report['score'] - last_score:+d})"
        heading(f"STEP {step} - {name}", f"{report['score']}/10  {verdict}{moved}")

        print(report["summary"])
        print(f"\nWhat it wants changed ({len(report['issues'])}):")
        for number, issue in enumerate(report["issues"], start=1):
            print(f"  {number}. [{issue['severity']}] {issue['criterion']}")
            print(textwrap.fill(issue["fix"], width=65,
                                initial_indent="     ", subsequent_indent="     "))
        print(f"\nLesson for next round: {report['lesson']}")

        if passed:
            break
        if last_score is not None and report["score"] <= last_score:
            print("\n(The score stopped going up, so we stop here.)")
            break
        last_score = report["score"]

        step += 1
        draft = rewrite(draft, report)
        heading(f"STEP {step} - THE WRITER REWRITES IT", f"{len(report['issues'])} changes")
        paragraph(draft)

    heading("HOW THE SCORE MOVED")
    for number, score in enumerate(scores, start=1):
        print(f"  round {number}   {score:>2}/10   {'#' * score}")

    if passed:
        heading(f"RESULT: PASSED - {scores[-1]}/10", "scores " + " -> ".join(str(s) for s in scores))
        print("\nFINAL DRAFT\n")
    else:
        heading(f"RESULT: NOT PASSED - best {max(scores)}/10, needed {PASS_MARK}/10",
                "scores " + " -> ".join(str(s) for s in scores))
        print("\nThe draft below is the rewrite made after the last score.")
        print("It was never scored - we ran out of rounds first.\n")
        print("DRAFT AS IT STANDS - NOT PASSED\n")

    paragraph(draft)
    return draft


print("improve() is ready.")

## Step 10 — Three drafts to try

Copy one into the box below. Each is bad in a **different** way, so each gets a different critique.

**1. All hype, no detail.**

```
AI is changing the world and you should learn it as soon as possible. Our course will make you skilled in all AI tools and help you get better opportunities. You will study Generative AI and Agentic AI with experienced trainers and projects. Join now to become future ready.
```

**2. Full of detail, but the numbers are made up.** The interesting one — invented figures *look* specific, so this scores well on specificity. The principles are what catch it.

```
Our graduates typically see a 40% salary increase within six months, and 9 out of 10 report better job satisfaction. Ranked the number one AI programme in the country, this weekend course guarantees placement support. Studies show professionals who learn AI now will earn double by 2027. Enrol today and transform your future.
```

**3. All true, but written for the wrong reader.**

```
Hey!! So basically we're running this SUPER cool weekend thing on GenAI and agents and it's gonna be lit. You'll learn loads about transformers, RAG pipelines, vector embeddings, and honestly way more than I can list here. Anyway lmk if you're keen!!
```

## Step 11 — Run it

Paste a draft and press Enter once.

Read the critic's list **before** you read the rewrite. That list is the real output of this session.

In [ ]:
draft = input("Paste your draft here, then press Enter:\n\n").strip()

if not draft:
    raise ValueError("Nothing was entered. Run this cell again and paste a draft.")

print(f"\n{len(draft.split())} words. Sending it to the critic...")

final_draft = improve(draft)

## Try these next

1. **Change `PASS_MARK` to 9** in Step 2 and run the same draft again. More rounds? A better answer, or just a longer one?
2. **Empty the checklist** (`CHECKLIST = ""`) and run again. You still get a score. What is it measuring now?
3. **Delete the principles** and run draft 2, the one with invented statistics. Watch which problems stop being reported.
4. **Show the checklist to the writer** by pasting `CHECKLIST` into `rewrite()`. Scores go up. Then read the draft.

## What to take away

- A review is only as good as the checklist you give it. "Make it better" gets you "it is better".
- Asking for JSON is what makes this automatic. A score your code can read is a score your code can act on.
- **Your code decides when to stop, never the model.**
- Rewriting fixes how something is *said*. It cannot supply a fact the model never had — which is exactly what a stuck score is telling you.